# Data Preparation

This notebook performs the initial data preparation stage for the project **“Emotional Characterization of Clinical Interview Transcripts in Spanish and English for the Detection of Mild Cognitive Impairment”**.

The goal of this stage is to inspect the available raw audio data, validate the dataset structure, review the ground truth file, and prepare a master metadata table that will support the following steps of the pipeline.

At this stage, the notebook focuses on:

- verifying whether the folder `/original-audio` is available,
- counting how many audio files are stored in that directory,
- identifying the audio formats present in the dataset,
- inspecting the contents of `groundtruth.csv`,
- detecting the language of each available audio file,
- selecting only recordings in Spanish or English for further analysis,
- and creating a master CSV file that will be progressively enriched throughout the project.

This master file will serve as the central reference table for tracking each sample across the complete workflow, including transcription, segmentation, emotion analysis, feature extraction, and classification.

## Raw Audio Directory Check

This section verifies whether the raw audio directory is available.  
The raw audio files were found in `../../../dataset-mineria/original-audio`, which will be used as the input directory for the next preparation steps.

In [4]:
from pathlib import Path

AUDIO_DIR = Path("../../../dataset-mineria/original-audio")

print("Audio directory:", AUDIO_DIR.resolve())
print("Exists:", AUDIO_DIR.exists())
print("Is directory:", AUDIO_DIR.is_dir())

Audio directory: /Users/armando/Developer/dataset-mineria/original-audio
Exists: True
Is directory: True


## Audio File Count and Format Identification

This section counts the number of audio files available in the raw audio directory and identifies the file formats present in the dataset.  

In [5]:
from pathlib import Path
from collections import Counter

AUDIO_DIR = Path("../../../dataset-mineria/original-audio")

# Extensiones de audio más comunes
AUDIO_EXTENSIONS = {
    ".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".wma", ".aiff", ".aif"
}

# Buscar archivos de audio
audio_files = [p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS]

# Contar formatos
format_counts = Counter(p.suffix.lower() for p in audio_files)

print("Audio directory:", AUDIO_DIR.resolve())
print("Exists:", AUDIO_DIR.exists())
print(f"Total audio files found: {len(audio_files)}")
print("\nAudio formats detected:")

for fmt, count in sorted(format_counts.items()):
    print(f"  {fmt}: {count}")

Audio directory: /Users/armando/Developer/dataset-mineria/original-audio
Exists: True
Total audio files found: 387

Audio formats detected:
  .wav: 387


## Ground Truth File Inspection

This section inspects the contents of `groundtruth.csv`, which contains the reference metadata associated with the audio recordings.  

In [6]:
from pathlib import Path
import pandas as pd

GROUNDTRUTH_PATH = Path("../../../dataset-mineria/groundtruth.csv")

print("Ground truth path:", GROUNDTRUTH_PATH.resolve())
print("Exists:", GROUNDTRUTH_PATH.exists())

df_gt = pd.read_csv(GROUNDTRUTH_PATH)

print("\nShape:")
print(df_gt.shape)

print("\nColumns:")
print(df_gt.columns.tolist())

print("\nFirst 5 rows:")
display(df_gt.head())

Ground truth path: /Users/armando/Developer/dataset-mineria/groundtruth.csv
Exists: True

Shape:
(387, 5)

Columns:
['tkdname', 'age', 'sex', 'mmse', 'dx']

First 5 rows:


,tkdname,age,sex,mmse,dx
0,taukdial-002-1.wav,72,F,29,NC
1,taukdial-002-2.wav,72,F,29,NC
2,taukdial-002-3.wav,72,F,29,NC
3,taukdial-003-1.wav,80,M,23,MCI
4,taukdial-003-2.wav,80,M,23,MCI


## Audio Language Detection

This section identifies the dominant language of each available audio recording.  
The goal is to keep only audio files in Spanish or English, since these are the two languages considered in the project.

In [7]:
from pathlib import Path
import pandas as pd
import whisper

# --------------------------------------------------
# Paths
# --------------------------------------------------
AUDIO_DIR = Path("../../../dataset-mineria/original-audio")

# --------------------------------------------------
# Collect audio files
# --------------------------------------------------
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".wma", ".aiff", ".aif"}
audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS])

print(f"Total audio files found: {len(audio_files)}")

# --------------------------------------------------
# Load Whisper model
# --------------------------------------------------
model = whisper.load_model("base")

# --------------------------------------------------
# Detect language for each audio
# --------------------------------------------------
rows = []

for i, audio_path in enumerate(audio_files, start=1):
    try:
        audio = whisper.load_audio(str(audio_path))
        audio = whisper.pad_or_trim(audio)

        mel = whisper.log_mel_spectrogram(audio).to(model.device)
        _, probs = model.detect_language(mel)

        detected_lang = max(probs, key=probs.get)
        lang_prob = probs[detected_lang]

    except Exception as e:
        detected_lang = None
        lang_prob = None
        print(f"[ERROR] {audio_path.name}: {e}")

    rows.append({
        "audio_file": audio_path.name,
        "audio_stem": audio_path.stem,
        "audio_path": str(audio_path),
        "detected_language": detected_lang,
        "language_probability": lang_prob
    })

    print(f"[{i}/{len(audio_files)}] {audio_path.name} -> {detected_lang} ({lang_prob})")

df_lang = pd.DataFrame(rows)

# --------------------------------------------------
# Keep only Spanish and English
# --------------------------------------------------
LANG_MAP = {
    "en": "English",
    "es": "Spanish"
}

df_lang["language_name"] = df_lang["detected_language"].map(LANG_MAP)
df_lang["is_target_language"] = df_lang["detected_language"].isin(["en", "es"])

print("\nDetected language counts:")
print(df_lang["detected_language"].value_counts(dropna=False))

print("\nTarget language counts (English/Spanish only):")
print(df_lang["language_name"].value_counts(dropna=False))

display(df_lang.head())

Total audio files found: 387
[1/387] taukdial-002-1.wav -> en (0.9992837309837341)
[2/387] taukdial-002-2.wav -> en (0.997840166091919)
[3/387] taukdial-002-3.wav -> en (0.9986321330070496)
[4/387] taukdial-003-1.wav -> zh (0.9693382978439331)
[5/387] taukdial-003-2.wav -> zh (0.9024975299835205)
[6/387] taukdial-003-3.wav -> zh (0.9006768465042114)
[7/387] taukdial-004-1.wav -> my (0.4197703003883362)
[8/387] taukdial-004-2.wav -> my (0.544195294380188)
[9/387] taukdial-004-3.wav -> my (0.4356679618358612)
[10/387] taukdial-005-1.wav -> en (0.9995948672294617)
[11/387] taukdial-005-2.wav -> en (0.9996440410614014)
[12/387] taukdial-005-3.wav -> en (0.9994773268699646)
[13/387] taukdial-006-1.wav -> zh (0.9926403164863586)
[14/387] taukdial-006-2.wav -> zh (0.987817108631134)
[15/387] taukdial-006-3.wav -> zh (0.9835669994354248)
[16/387] taukdial-007-1.wav -> zh (0.9308678507804871)
[17/387] taukdial-007-2.wav -> zh (0.9637993574142456)
[18/387] taukdial-007-3.wav -> zh (0.94865369796

,audio_file,audio_stem,audio_path,detected_language,language_probability,language_name,is_target_language
0,taukdial-002-1.wav,taukdial-002-1,../../../dataset-mineria/original-audio/taukdi...,en,0.999284,English,True
1,taukdial-002-2.wav,taukdial-002-2,../../../dataset-mineria/original-audio/taukdi...,en,0.997840,English,True
2,taukdial-002-3.wav,taukdial-002-3,../../../dataset-mineria/original-audio/taukdi...,en,0.998632,English,True
3,taukdial-003-1.wav,taukdial-003-1,../../../dataset-mineria/original-audio/taukdi...,zh,0.969338,NaN,False
4,taukdial-003-2.wav,taukdial-003-2,../../../dataset-mineria/original-audio/taukdi...,zh,0.902498,NaN,False


## Master Metadata Initialization

At this stage, the project begins the construction of a master metadata table.  
This table integrates the available audio file information, the detected language for each recording, and the reference information provided in `groundtruth.csv`.

The master CSV will be progressively enriched in later stages of the pipeline, serving as the central file for traceability and analysis.

In [12]:
from pathlib import Path
import pandas as pd
import whisper

# ==================================================
# Paths
# ==================================================
DATASET_DIR = Path("../../../dataset-mineria")
AUDIO_DIR = DATASET_DIR / "original-audio"
GROUNDTRUTH_PATH = DATASET_DIR / "groundtruth.csv"
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV_PATH = OUTPUT_DIR / "master_metadata.csv"

# ==================================================
# Load ground truth
# ==================================================
df_gt = pd.read_csv(GROUNDTRUTH_PATH)

# Clean merge key in ground truth
df_gt = df_gt.copy()
df_gt["audio_file"] = df_gt["tkdname"].astype(str).str.strip()

# ==================================================
# Collect audio files
# ==================================================
AUDIO_EXTENSIONS = {
    ".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".wma", ".aiff", ".aif"
}

audio_files = sorted([
    p for p in AUDIO_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
])

print(f"Total audio files found: {len(audio_files)}")

# ==================================================
# Load Whisper model
# ==================================================
model = whisper.load_model("base")

# ==================================================
# Detect language and build audio table
# ==================================================
rows = []

for i, audio_path in enumerate(audio_files, start=1):
    try:
        audio = whisper.load_audio(str(audio_path))
        audio = whisper.pad_or_trim(audio)

        mel = whisper.log_mel_spectrogram(audio).to(model.device)
        _, probs = model.detect_language(mel)

        detected_lang = max(probs, key=probs.get)
        lang_prob = probs[detected_lang]

    except Exception as e:
        detected_lang = None
        lang_prob = None
        print(f"[ERROR] {audio_path.name}: {e}")

    rows.append({
        "audio_file": audio_path.name,
        "audio_stem": audio_path.stem,
        "audio_path": str(audio_path),
        "audio_extension": audio_path.suffix.lower(),
        "detected_language": detected_lang,
        "language_probability": lang_prob,
        "language_name": {
            "en": "English",
            "es": "Spanish",
            "zh": "Chinese",
            "my": "Burmese"
        }.get(detected_lang, "Other"),
        "is_target_language": detected_lang in {"en", "es"}
    })

    print(f"[{i}/{len(audio_files)}] {audio_path.name} -> {detected_lang} ({lang_prob})")

df_audio = pd.DataFrame(rows)

# ==================================================
# Merge with ground truth using full file name
# ==================================================
master_df = df_audio.merge(df_gt, on="audio_file", how="left")

# ==================================================
# Save master CSV
# ==================================================
master_df.to_csv(MASTER_CSV_PATH, index=False)

print("\nSaved:", MASTER_CSV_PATH.resolve())
print("Shape:", master_df.shape)

print("\nPreview of merged columns:")
print(master_df[["audio_file", "tkdname", "age", "sex", "mmse", "dx"]].head(10))

Total audio files found: 387
[1/387] taukdial-002-1.wav -> en (0.9992837309837341)
[2/387] taukdial-002-2.wav -> en (0.997840166091919)
[3/387] taukdial-002-3.wav -> en (0.9986321330070496)
[4/387] taukdial-003-1.wav -> zh (0.9693382978439331)
[5/387] taukdial-003-2.wav -> zh (0.9024975299835205)
[6/387] taukdial-003-3.wav -> zh (0.9006768465042114)
[7/387] taukdial-004-1.wav -> my (0.4197703003883362)
[8/387] taukdial-004-2.wav -> my (0.544195294380188)
[9/387] taukdial-004-3.wav -> my (0.4356679618358612)
[10/387] taukdial-005-1.wav -> en (0.9995948672294617)
[11/387] taukdial-005-2.wav -> en (0.9996440410614014)
[12/387] taukdial-005-3.wav -> en (0.9994773268699646)
[13/387] taukdial-006-1.wav -> zh (0.9926403164863586)
[14/387] taukdial-006-2.wav -> zh (0.987817108631134)
[15/387] taukdial-006-3.wav -> zh (0.9835669994354248)
[16/387] taukdial-007-1.wav -> zh (0.9308678507804871)
[17/387] taukdial-007-2.wav -> zh (0.9637993574142456)
[18/387] taukdial-007-3.wav -> zh (0.94865369796